# 🔊 NEURO BAND — Voice Fix (gTTS)
### Replaces the old Web Speech cells — this one actually plays audio in Colab

---
**Why the old version had no sound:**  
Colab runs inside a sandboxed `<iframe>` that blocks the browser's Web Speech API audio.  

**The fix:**  
Use **gTTS** (Google Text-to-Speech) → it generates a real `.mp3` → plays it with an audio widget.  
✅ 100% free &nbsp;|&nbsp; ✅ No API key &nbsp;|&nbsp; ✅ Works in Colab &nbsp;|&nbsp; ✅ Clear natural voice

**Run order:** Cell 7-A → Cell 7-B → Cell 8

In [ ]:
# ============================================================
# CELL 7-A  —  INSTALL gTTS  (run once per session)
# ============================================================

!pip install gtts -q

from gtts import gTTS
from IPython.display import display, Audio, HTML
import os, re, textwrap

# ── Voice language setting ───────────────────────────────────
# Full list: https://gtts.readthedocs.io/en/latest/module.html#languages-gtts-lang
VOICE_LANG  = 'en'     # 'en' = English (clear & natural)
VOICE_SLOW  = False    # True = slower, easier to follow
TTS_FILE    = '/content/neuro_brief_voice.mp3'

print('✅ gTTS installed and ready!')
print(f'   Language : {VOICE_LANG}')
print(f'   Slow mode: {VOICE_SLOW}')
print()
print('👉 Now run Cell 7-B to generate and play your health brief.')

In [ ]:
# ============================================================
# CELL 7-B  —  SPEAK THE HEALTH BRIEF
# Generates MP3 with gTTS and plays it inline
# ============================================================

def clean_for_speech(text):
    """Strip emoji, markdown symbols, and extra spaces for clean TTS."""
    # Remove emoji (broad unicode range)
    text = re.sub(
        r'[\U0001F300-\U0001FAFF\U00002702-\U000027B0\U0001F000-\U0001F02F'
        r'\U0001F0A0-\U0001F0FF\U0001F100-\U0001F1FF\U0001F200-\U0001F2FF'
        r'\U0001F004\U0001F0CF\U0001F170-\U0001F171\U0001F17E-\U0001F17F'
        r'\u2600-\u26FF\u2700-\u27BF]+', '', text)
    # Remove markdown-style symbols
    text = re.sub(r'[\*\_\#\`\~\>\|]', '', text)
    # Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def speak_health_brief_gtts(recs_text=None):
    """
    Convert the AI health brief to speech using gTTS and play it.
    Pass recs_text manually, or leave None to auto-use health_recs.
    """
    # ── 1. Get the text ──────────────────────────────────────
    try:
        raw = recs_text or health_recs or ''
    except NameError:
        raw = ''

    if not raw:
        print('⚠️  No health recommendations found.')
        print('   Please run Cell 7 first, then re-run this cell.')
        return

    # ── 2. Build the spoken script ───────────────────────────
    try:   name = YOUR_NAME
    except NameError: name = 'friend'

    try:   city = weather_info.get('city', 'your city')
    except NameError: city = 'your city'

    try:   temp = weather_info.get('temp_c', '?')
    except NameError: temp = '?'

    try:   cond = weather_info.get('desc', '')
    except NameError: cond = ''

    intro = (
        f"Good morning, {name}! "
        f"Welcome to your Neuro Band Morning Brief. "
        f"Today in {city}, it is {temp} degrees celsius with {cond}. "
        f"Here are your personalised health recommendations for today. "
    )
    outro = (
        " That's your health brief for today. "
        "Stay focused, drink plenty of water, and have a great day!"
    )

    full_script = intro + clean_for_speech(str(raw)) + outro

    print('📝 Script preview (first 200 chars):')
    print('  ', full_script[:200], '...')
    print()

    # ── 3. Generate MP3 ──────────────────────────────────────
    print('🎙️  Generating voice audio with gTTS...')
    try:
        tts = gTTS(text=full_script, lang=VOICE_LANG, slow=VOICE_SLOW)
        tts.save(TTS_FILE)
        size_kb = os.path.getsize(TTS_FILE) // 1024
        print(f'✅ Audio generated!  ({size_kb} KB)  →  {TTS_FILE}')
    except Exception as e:
        print(f'❌ gTTS error: {e}')
        print('   Check your internet connection and try again.')
        return

    # ── 4. Display the player card ───────────────────────────
    display(HTML("""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@400;500&display=swap');
    </style>
    <div style='
      background: linear-gradient(145deg,#0f0c29,#302b63);
      border: 1px solid rgba(255,255,255,.1);
      border-radius: 18px;
      padding: 22px 24px;
      max-width: 560px;
      box-shadow: 0 20px 60px rgba(0,0,0,.5);
      font-family: DM Sans, sans-serif;
      color: white;
    '>
      <div style='display:flex;align-items:center;gap:12px;margin-bottom:14px;'>
        <span style='font-size:2em;'>🔊</span>
        <div>
          <div style='font-family:Syne,sans-serif;font-weight:800;font-size:1.05em;
                      letter-spacing:2px;'>NEURO BAND VOICE</div>
          <div style='color:#7b8cde;font-size:.78em;'>Health Brief — spoken by gTTS</div>
        </div>
        <span style='margin-left:auto;background:rgba(46,213,115,.2);
                     color:#2ed573;padding:3px 10px;border-radius:20px;font-size:.75em;'>
          ✅ Audio Ready
        </span>
      </div>
      <div style='background:rgba(255,255,255,.05);border-radius:10px;
                  padding:10px 14px;margin-bottom:14px;
                  border-left:3px solid #667eea;font-size:.82em;
                  color:#c5cae9;line-height:1.6;'>
        Press ▶ on the audio player below to hear your brief.
      </div>
    </div>
    """))

    # ── 5. Play the audio inline ─────────────────────────────
    display(Audio(TTS_FILE, autoplay=True))

    print()
    print('🔊 Audio player shown above — press ▶ to play!')
    print('   Tip: If autoplay is blocked, just click the ▶ button manually.')


# ── Run it ────────────────────────────────────────────────────
speak_health_brief_gtts()
print('\n👉 Now run Cell 8 for the full dashboard with built-in voice button!')

In [ ]:
# ============================================================
# CELL 8  —  MORNING BRIEF DASHBOARD  (with 🔊 Speak button)
# This replaces the old Cell 8 entirely
# ============================================================

import ipywidgets as widgets
from IPython.display import display, HTML, Audio, clear_output
from datetime import datetime, date, timedelta
import os, re

def generate_morning_brief_v2():
    data       = load_data()
    today_date = datetime.now().strftime('%A, %B %d, %Y')
    today_time = datetime.now().strftime('%I:%M %p')

    yest_cards = data['flashcards'].get(yesterday_str, [])
    pending    = [t for t in data['todos'] if not t['done']]
    done_tasks = [t for t in data['todos'] if  t['done']]
    yest_topics= list(set(c.get('topic','General') for c in yest_cards))

    w = weather_info

    # ── Weather grid ─────────────────────────────────────────
    weather_html = f"""
    <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin:10px 0;'>
      {''.join([
        f"<div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>"
        f"<div style='font-size:1.5em'>{icon}</div>"
        f"<div style='font-size:1em;font-weight:700'>{val}</div>"
        f"<div style='opacity:.55;font-size:.72em'>{label}</div></div>"
        for icon, val, label in [
            ('🌡️', f"{w['temp_c']}°C", f"feels {w['feels_c']}°C"),
            ('💧', f"{w['humidity']}%", 'Humidity'),
            ('☀️', w['uv_index'],       'UV Index'),
            ('💨', f"{w['wind_kmph']} km/h", 'Wind'),
        ]
      ])}
    </div>
    <div style='text-align:center;opacity:.65;font-size:.82em;'>
      {w['desc']} &nbsp;|&nbsp; {w['min_c']}°C – {w['max_c']}°C today
    </div>"""

    # ── Flashcards ────────────────────────────────────────────
    if yest_cards:
        items = ''.join([
            f"<li style='margin:5px 0'>"
            f"<span style='background:rgba(102,126,234,.45);padding:1px 7px;"
            f"border-radius:10px;font-size:.74em;'>{c.get('topic','?')}</span> "
            f"{c['question'][:55]}{'…' if len(c['question'])>55 else ''}</li>"
            for c in yest_cards[:6]
        ])
        extra   = f"<li style='opacity:.45'>…and {len(yest_cards)-6} more</li>" if len(yest_cards)>6 else ""
        fc_html = f"<ul style='margin:6px 0;padding-left:18px;'>{items}{extra}</ul>"
    else:
        fc_html = "<p style='opacity:.45;margin:6px 0;'>No flashcards yesterday — add some in Cell 4!</p>"

    # ── To-do ─────────────────────────────────────────────────
    if pending:
        prio_order  = {'🔴 High':0,'🟡 Medium':1,'🟢 Low':2}
        badge_color = {'🔴 High':'#ff4757','🟡 Medium':'#ffa502','🟢 Low':'#2ed573'}
        sp = sorted(pending, key=lambda t: prio_order.get(t['priority'],1))
        todo_items = ''.join([
            f"<li style='margin:5px 0;'>"
            f"<span style='background:{badge_color.get(t['priority'],'#888')};"
            f"padding:1px 7px;border-radius:10px;font-size:.74em;color:#fff;'>"
            f"{t['priority'].split()[-1].upper()}</span> {t['task']}</li>"
            for t in sp[:8]
        ])
        todo_html = f"<ul style='margin:6px 0;padding-left:18px;'>{todo_items}</ul>"
    else:
        todo_html = "<p style='opacity:.45;margin:6px 0;'>All caught up! No pending tasks.</p>"

    # ── Health text ───────────────────────────────────────────
    health_html = str(health_recs).replace('\n','<br>') if health_recs else \
        'Run Cell 7 to generate recommendations.'

    # ── Render the dashboard ──────────────────────────────────
    display(HTML(f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap');
      .nb2 * {{ font-family:'DM Sans',sans-serif; color:white; box-sizing:border-box; }}
      .nb2 h1 {{ font-family:'Syne',sans-serif; }}
      .sec {{ background:rgba(255,255,255,.06); border-radius:14px;
               padding:14px 16px; margin-bottom:14px; }}
      .lbl {{ font-size:.73em; letter-spacing:2px; opacity:.5;
               font-family:'Syne',sans-serif; text-transform:uppercase; margin-bottom:6px; }}
    </style>

    <div class='nb2' style='
      background:linear-gradient(145deg,#0f0c29 0%,#302b63 55%,#1a1a2e 100%);
      padding:28px 24px; border-radius:22px; max-width:720px;
      border:1px solid rgba(255,255,255,.08);
      box-shadow:0 30px 80px rgba(0,0,0,.6);
    '>
      <div style='text-align:center;margin-bottom:22px;
                  padding-bottom:18px;border-bottom:1px solid rgba(255,255,255,.1);'>
        <div style='font-size:2.6em;margin-bottom:4px;'>🧠⚡</div>
        <h1 style='margin:0;font-size:1.9em;font-weight:800;letter-spacing:3px;'>NEURO BAND</h1>
        <p style='margin:3px 0 0;opacity:.4;font-size:.78em;letter-spacing:4px;'>MORNING BRIEF SYSTEM</p>
        <div style='margin-top:12px;display:flex;justify-content:center;gap:20px;
                    opacity:.65;font-size:.83em;flex-wrap:wrap;'>
          <span>👋 Good morning, {YOUR_NAME}!</span>
          <span>📅 {today_date}</span>
          <span>⏰ {today_time}</span>
        </div>
        <div style='margin-top:10px;display:flex;justify-content:center;gap:12px;flex-wrap:wrap;'>
          <span style='background:rgba(102,126,234,.3);padding:3px 12px;border-radius:20px;font-size:.78em;'>📚 {len(yest_cards)} cards yesterday</span>
          <span style='background:rgba(17,153,142,.3);padding:3px 12px;border-radius:20px;font-size:.78em;'>✅ {len(pending)} pending</span>
          <span style='background:rgba(255,71,87,.3);padding:3px 12px;border-radius:20px;font-size:.78em;'>🏆 {len(done_tasks)} done</span>
        </div>
      </div>

      <div class='sec'><div class='lbl'>🌤️ Weather — {w['city']}</div>{weather_html}</div>
      <div class='sec'><div class='lbl'>📚 Yesterday's Flashcards — {len(yest_cards)} | {', '.join(yest_topics) if yest_topics else 'No topics'}</div>{fc_html}</div>
      <div class='sec'><div class='lbl'>✅ To-Do — {len(pending)} pending / {len(done_tasks)} done</div>{todo_html}</div>

      <div style='background:rgba(102,126,234,.15);border:1px solid rgba(102,126,234,.35);
                  border-radius:14px;padding:14px 16px;margin-bottom:14px;'>
        <div class='lbl'>🧬 AI Health Brief — HuggingFace</div>
        <div style='line-height:1.75;font-size:.87em;opacity:.88;'>{health_html}</div>
      </div>

      <div style='text-align:center;opacity:.22;font-size:.68em;letter-spacing:2px;'>
        NEURO BAND MORNING BRIEF • {today_date}
      </div>
    </div>
    """))

    # ── Voice button (widget, always works) ──────────────────
    voice_out = widgets.Output()

    speak_btn = widgets.Button(
        description  = '🔊  Speak Health Brief',
        button_style = 'primary',
        layout       = widgets.Layout(width='220px', height='40px')
    )

    def on_speak(b):
        with voice_out:
            clear_output()
            print('🎙️  Generating audio...')
            try:
                raw = str(health_recs) if health_recs else 'No health data.'
                intro = (
                    f"Good morning, {YOUR_NAME}! Here is your health brief. "
                    f"Today in {w.get('city','your city')}, "
                    f"it is {w.get('temp_c','?')} degrees celsius with {w.get('desc','')}. "
                    f"Here are your health recommendations. "
                )
                outro = " Have a wonderful and productive day!"
                # Clean emoji from raw text
                clean = re.sub(
                    r'[\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]+', '', raw)
                clean = re.sub(r'\s+', ' ', clean).strip()
                script = intro + clean + outro

                tts = gTTS(text=script, lang=VOICE_LANG, slow=VOICE_SLOW)
                tts.save(TTS_FILE)
                print('✅ Playing now — look for the audio player below!')
                display(Audio(TTS_FILE, autoplay=True))
            except Exception as e:
                print(f'❌ Error: {e}')

    speak_btn.on_click(on_speak)

    display(
        widgets.HTML(value="<div style='margin:10px 0 4px;font-family:sans-serif;"
                           "color:#888;font-size:.82em;'>🔊 Click to hear your health brief spoken aloud:</div>"),
        speak_btn,
        voice_out
    )

generate_morning_brief_v2()

---
## 🛠️ Troubleshooting

| Problem | Fix |
|---------|-----|
| **No sound at all** | Click the ▶ button on the audio player manually (browser blocks autoplay) |
| **`gTTS` error** | Check your internet. Colab needs internet to call Google TTS |
| **`health_recs` not defined** | Run Cell 7 first, then re-run Cell 7-B |
| **Audio cuts off** | Run Cell 7-B again — sometimes Colab needs a second try |
| **Want slower speech** | In Cell 7-A, change `VOICE_SLOW = True` and re-run |

### 🌍 Change Language / Accent
In Cell 7-A, change `VOICE_LANG`:
```python
VOICE_LANG = 'en'     # English (default)
VOICE_LANG = 'en-uk'  # British English
VOICE_LANG = 'en-au'  # Australian English
VOICE_LANG = 'en-in'  # Indian English
```